# Block 2 — live lecture demo

Instructor notebook for the 15-minute Block 2 lecture. Run top to bottom.

The root, `Study`, `MetaDataVersion` and `Standards` are pre-built in the setup
cell below — exactly as they are given to you in `create_define_exercise.ipynb`.
We live-build the interesting part: the dataset, its variables, the codelist.

## 0 · Setup — given, not taught

In [ ]:
import os
import odmlib.define_2_1.model as DEF

os.makedirs("output", exist_ok=True)

odm = DEF.ODM(
    FileOID="DEF.RPH2026.DM", FileType="Snapshot",
    CreationDateTime="2026-08-19T12:00:00", ODMVersion="1.3.2",
    Context="Submission",                       # def:Context - required in v2.1
    Originator="R/Pharma 2026 Workshop", SourceSystem="odmlib",
)
study = DEF.Study(OID="ST.RPH2026")
study.GlobalVariables = DEF.GlobalVariables(
    StudyName=DEF.StudyName(_content="RPH2026"),
    StudyDescription=DEF.StudyDescription(_content="R/Pharma 2026 odmlib workshop study"),
    ProtocolName=DEF.ProtocolName(_content="RPH-2026-001"),
)
mdv = DEF.MetaDataVersion(
    OID="MDV.RPH2026.1", Name="RPH2026 Data Definitions",
    Description="Demographics metadata for the workshop", DefineVersion="2.1.0",
)
standards = DEF.Standards()
standards.Standard.append(
    DEF.Standard(OID="STD.1", Name="SDTMIG", Type="IG", Version="3.4", Status="Final"))
mdv.Standards = standards

print("given:", odm.FileOID, "|", study.OID, "|", mdv.OID)

## 1 · Required attributes are enforced at construction

Not three steps later, not at serialization — *now*.

In [ ]:
from odmlib.exceptions import OdmlibRequiredAttributeError

try:
    DEF.ItemGroupDef(OID="IG.DM", Name="DM")        # no Repeating
except OdmlibRequiredAttributeError as e:
    print("caught at __init__:", e)

## 2 · The DM dataset — `ItemGroupDef`

In [ ]:
igd = DEF.ItemGroupDef(
    OID="IG.DM", Name="DM", Repeating="No", IsReferenceData="No",
    SASDatasetName="DM", Domain="DM", Purpose="Tabulation",
    Structure="One record per subject",
    ArchiveLocationID="LF.DM",        # must match the leaf ID below
    StandardOID="STD.1",              # ties the dataset to a declared Standard
)

igd.Description = DEF.Description()                            # single child: assign
igd.Description.TranslatedText.append(                         # list child: append
    DEF.TranslatedText(_content="Demographics", lang="en"))

for oid, mand, order, key in [("IT.DM.STUDYID", "Yes", 1, 1), ("IT.DM.USUBJID", "Yes", 2, 2),
                              ("IT.DM.AGE", "No", 3, None), ("IT.DM.SEX", "Yes", 4, None)]:
    kw = {"ItemOID": oid, "Mandatory": mand, "OrderNumber": order}
    if key:
        kw["KeySequence"] = key
    igd.ItemRef.append(DEF.ItemRef(**kw))

igd.Class = DEF.Class(Name="SPECIAL PURPOSE")
igd.leaf = DEF.leaf(ID="LF.DM", href="dm.xpt", title=DEF.title(_content="dm.xpt"))

print(f"{igd.Name}: {len(igd.ItemRef)} ItemRefs | leaf {igd.leaf.ID} <-> {igd.ArchiveLocationID}")

Children went in **schema order**: `Description`, `ItemRef`s, `Class`, `leaf`.

`href` is an `xlink:href` and `title` is a `def:title` — you never typed a prefix.

## 3 · The variables

A helper bakes in the right child order (`Description`, `CodeListRef`, `Origin`)
once — worth copying for real projects with hundreds of variables.

In [ ]:
def make_item(oid, name, dtype, length, desc, codelist=None):
    item = DEF.ItemDef(OID=oid, Name=name, DataType=dtype, Length=length, SASFieldName=name)
    item.Description = DEF.Description()
    item.Description.TranslatedText.append(DEF.TranslatedText(_content=desc, lang="en"))
    if codelist:
        item.CodeListRef = DEF.CodeListRef(CodeListOID=codelist)
    item.Origin.append(DEF.Origin(Type="Collected"))
    return item

mdv.ItemDef.append(make_item("IT.DM.STUDYID", "STUDYID", "text", 12, "Study Identifier"))
mdv.ItemDef.append(make_item("IT.DM.USUBJID", "USUBJID", "text", 25, "Unique Subject Identifier"))
mdv.ItemDef.append(make_item("IT.DM.AGE", "AGE", "integer", 3, "Age"))
mdv.ItemDef.append(make_item("IT.DM.SEX", "SEX", "text", 1, "Sex", codelist="CL.SEX"))

print(f"{len(mdv.ItemDef)} ItemDefs")

## 4 · The SEX codelist — wired up by that `CodeListRef`

In [ ]:
cl = DEF.CodeList(OID="CL.SEX", Name="Sex", DataType="text")
for coded, decode in (("F", "Female"), ("M", "Male")):
    term = DEF.CodeListItem(CodedValue=coded)
    term.Decode = DEF.Decode()
    term.Decode.TranslatedText.append(DEF.TranslatedText(_content=decode, lang="en"))
    cl.CodeListItem.append(term)
mdv.CodeList.append(cl)

print(cl.OID, "->", [t.CodedValue for t in cl.CodeListItem])

## 5 · Assemble and write

`Study` and `MetaDataVersion` are single objects — **assign**, don't append.

In [ ]:
mdv.ItemGroupDef.append(igd)
study.MetaDataVersion = mdv      # assignment - single object
odm.Study = study                # assignment - single object

odm.write_xml("output/define_dm.xml")
print("wrote output/define_dm.xml")

Four namespaces, schema order, correct prefixes — none of which we thought about:

In [ ]:
print(open("output/define_dm.xml").read()[:900])

### `to_xml_string()` on any element — the debugging tool to reach for

In [ ]:
print(igd.leaf.to_xml_string())

### And the references already resolve

In [ ]:
from odmlib import create_oid_checker

odm.verify_oids(create_oid_checker("define_2_1"))
print("all OID references resolve — that is Block 3's layer 2, arriving early")

---

**Your turn:** `create_define_exercise.ipynb` — 4 TODOs, ~20 minutes.
Stretch goal adds value-level metadata (`ValueListDef` + `WhereClauseDef`).